In [11]:
import torch
from torch import nn

In [12]:
def tensor_memory_megabytes(
    tensor: torch.Tensor,
) -> float:
    """Tensor 하나가 차지하는 raw memory를 MiB 단위로 계산"""
    
    memory_bytes = (
        tensor.numel()
        * tensor.element_size()
    )

    memory_megabytes = (
        memory_bytes
        / 1024**2
    )

    return memory_megabytes


# 2D image contract: [B, C, H, W]
input_2d = torch.randn(
    1,
    1,
    64,
    64,
)
# 3D volume contract: [B, C, D, H, W]
input_3d = torch.randn(
    1,
    1,
    32,
    64,
    64,
)
convolution_2d = nn.Conv2d(
    in_channels=1,
    out_channels=8,
    kernel_size=3,
    padding=1,
)
convolution_3d = nn.Conv3d(
    in_channels=1,
    out_channels=8,
    kernel_size=3,
    padding=1,
)
output_2d = convolution_2d(
    input_2d,
)
output_3d = convolution_3d(
    input_3d,
)

print("2D input shape: ", input_2d.shape)
print("2D output shape:", output_2d.shape)
print("2D weight shape:", convolution_2d.weight.shape)
print("2D output memory:", tensor_memory_megabytes(output_2d), "MiB")

print()

print("3D input shape: ", input_3d.shape)
print("3D output shape:", output_3d.shape)
print("3D weight shape:", convolution_3d.weight.shape)
print("3D output memory:", tensor_memory_megabytes(output_3d), "MiB")

memory_ratio = (
    tensor_memory_megabytes(output_3d)
    / tensor_memory_megabytes(output_2d)
)

print()
print("3D / 2D output memory ratio:", memory_ratio)

2D input shape:  torch.Size([1, 1, 64, 64])
2D output shape: torch.Size([1, 8, 64, 64])
2D weight shape: torch.Size([8, 1, 3, 3])
2D output memory: 0.125 MiB

3D input shape:  torch.Size([1, 1, 32, 64, 64])
3D output shape: torch.Size([1, 8, 32, 64, 64])
3D weight shape: torch.Size([8, 1, 3, 3, 3])
3D output memory: 4.0 MiB

3D / 2D output memory ratio: 32.0


In [13]:
def estimate_tensor_memory_megabytes(
    tensor_shape: tuple[int, ...],
    bytes_per_element: int,
) -> float:
    """Tensor를 생성하지 않고 shape와 dtype 크기로 memory를 추정"""

    number_of_elements = 1

    for dimension_size in tensor_shape:
        number_of_elements *= dimension_size

    memory_bytes = (
        number_of_elements
        * bytes_per_element
    )

    return memory_bytes / 1024**2


# 각 shape는 Conv3d activation [B, C, D, H, W]
activation_shapes: list[
    tuple[str, tuple[int, ...]]
] = [
    (
        "Base",
        (1, 32, 32, 64, 64),
    ),
    (
        "Double batch",
        (2, 32, 32, 64, 64),
    ),
    (
        "Double depth",
        (1, 32, 64, 64, 64),
    ),
    (
        "Double H and W",
        (1, 32, 32, 128, 128),
    ),
    (
        "Double all spatial axes",
        (1, 32, 64, 128, 128),
    ),
]

for configuration_name, activation_shape in activation_shapes:
    float32_memory = estimate_tensor_memory_megabytes(
        tensor_shape=activation_shape,
        bytes_per_element=4,
    )

    float16_memory = estimate_tensor_memory_megabytes(
        tensor_shape=activation_shape,
        bytes_per_element=2,
    )

    print(
        f"{configuration_name:24s} | "
        f"shape={activation_shape} | "
        f"FP32={float32_memory:7.1f} MiB | "
        f"FP16={float16_memory:7.1f} MiB"
    )

Base                     | shape=(1, 32, 32, 64, 64) | FP32=   16.0 MiB | FP16=    8.0 MiB
Double batch             | shape=(2, 32, 32, 64, 64) | FP32=   32.0 MiB | FP16=   16.0 MiB
Double depth             | shape=(1, 32, 64, 64, 64) | FP32=   32.0 MiB | FP16=   16.0 MiB
Double H and W           | shape=(1, 32, 32, 128, 128) | FP32=   64.0 MiB | FP16=   32.0 MiB
Double all spatial axes  | shape=(1, 32, 64, 128, 128) | FP32=  128.0 MiB | FP16=   64.0 MiB


In [14]:
# [B, 1, 32, 64, 64]
#           │ DoubleConv3d
#           ▼
# [B, 8, 32, 64, 64] ─── skip feature: 4.0 MiB
#           │ MaxPool3d(2)
#           ▼
# [B, 8, 16, 32, 32] ─── pooled feature: 0.5 MiB


class DoubleConvolution3D(nn.Module):
    """Spatial size를 유지하면서 두 번의 Conv3d를 적용"""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        return self.layers(input_tensor)


class EncoderBlock3D(nn.Module):
    """3D feature를 추출하고 D, H, W를 각각 절반으로 축소"""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()

        self.feature_extractor = DoubleConvolution3D(
            in_channels=in_channels,
            out_channels=out_channels,
        )

        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        skip_features = self.feature_extractor(
            input_tensor,
        )

        pooled_features = self.pool(
            skip_features,
        )

        return skip_features, pooled_features


encoder_3d = EncoderBlock3D(
    in_channels=1,
    out_channels=8,
)

volume_input = torch.randn(
    1,
    1,
    32,
    64,
    64,
)

skip_features_3d, pooled_features_3d = encoder_3d(
    volume_input,
)

print(
    "Input: ",
    volume_input.shape,
    "|",
    tensor_memory_megabytes(volume_input),
    "MiB",
)

print(
    "Skip:  ",
    skip_features_3d.shape,
    "|",
    tensor_memory_megabytes(skip_features_3d),
    "MiB",
)

print(
    "Pooled:",
    pooled_features_3d.shape,
    "|",
    tensor_memory_megabytes(pooled_features_3d),
    "MiB",
)

Input:  torch.Size([1, 1, 32, 64, 64]) | 0.5 MiB
Skip:   torch.Size([1, 8, 32, 64, 64]) | 4.0 MiB
Pooled: torch.Size([1, 8, 16, 32, 32]) | 0.5 MiB


In [ ]:
def bytes_to_megabytes(
    number_of_bytes: int,
) -> float:
    """Byte를 MiB 단위로 변환"""

    return number_of_bytes / 1024**2


if not torch.cuda.is_available():
    raise RuntimeError(
        "이 cell은 CUDA GPU memory 측정이 필요합니다."
    )

cuda_device = torch.device("cuda")

gpu_encoder = EncoderBlock3D(
    in_channels=1,
    out_channels=8,
).to(cuda_device)

gpu_volume = torch.randn(
    1,
    1,
    32,
    64,
    64,
    device=cuda_device,
)

# 비동기 CUDA 연산이 모두 끝날 때까지 기다린다.
torch.cuda.synchronize(cuda_device)

baseline_memory = torch.cuda.memory_allocated(
    cuda_device,
)

torch.cuda.reset_peak_memory_stats(
    cuda_device,
)

gpu_encoder.train()

skip_features, pooled_features = gpu_encoder(
    gpu_volume,
)

# 학습 가능한 scalar loss를 임시로 만든다.
training_loss = (
    skip_features.square().mean()
    + pooled_features.square().mean()
)

training_loss.backward()

torch.cuda.synchronize(cuda_device)

peak_allocated_memory = torch.cuda.max_memory_allocated(
    cuda_device,
)

peak_reserved_memory = torch.cuda.max_memory_reserved(
    cuda_device,
)

additional_training_memory = (
    peak_allocated_memory
    - baseline_memory
)

raw_output_memory = (
    tensor_memory_megabytes(skip_features)
    + tensor_memory_megabytes(pooled_features)
)

print(
    "GPU:",
    torch.cuda.get_device_name(cuda_device),
)

print(
    "Input shape:",
    gpu_volume.shape,
)

print(
    "Raw returned outputs:", # 직접 받은 skip_features + pooled_features 크기
    raw_output_memory,
    "MiB",
)

print(
    "Baseline allocated:", # model과 input을 GPU에 올린 직후 사용량
    bytes_to_megabytes(baseline_memory),
    "MiB",
)

print(
    "Additional training peak:", # forward와 backward 중 추가로 실제 사용한 memory
    bytes_to_megabytes(additional_training_memory),
    "MiB",
)

print(
    "Peak allocated:", # baseline까지 포함한 최대 실사용량
    bytes_to_megabytes(peak_allocated_memory),
    "MiB",
)

print(
    "Peak reserved:", # PyTorch가 재사용하려고 GPU에서 확보해 둔 memory
    bytes_to_megabytes(peak_reserved_memory),
    "MiB",
)

GPU: NVIDIA GeForce RTX 3060 Ti
Input shape: torch.Size([1, 1, 32, 64, 64])
Raw returned outputs: 4.5 MiB
Baseline allocated: 0.5126953125 MiB
Additional training peak: 34.0029296875 MiB
Peak allocated: 34.515625 MiB
Peak reserved: 46.0 MiB
